# QASPER exploratory data analysis

Explore the original train/dev datasets and, when available, the processed corpus
and evidence mappings. This notebook reads files without modifying them.

Run the cells in order using the repository's `.venv` kernel. Install
`ipykernel matplotlib` in that environment first. Raw QASPER v0.3 train/dev JSON
files must be under `data/raw/qasper/`. Optional processed files are read from
`data/processed/qasper/{split}/`.

The examples use train data. Train/dev comparisons below are descriptive data
checks, not retrieval results or threshold-tuning experiments.


In [ ]:
import json
import math
from collections import Counter
from itertools import islice
from pathlib import Path
from statistics import mean, median

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Locate the repository whether this kernel starts in the root or notebooks/.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts/prepare_qasper.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Start the notebook from the repository root or notebooks/.")

RAW_DIR = ROOT / "data/raw/qasper"
PROCESSED_DIR = ROOT / "data/processed/qasper"
SPLITS = ("train", "dev")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Repository:", ROOT)


In [ ]:
def load_json(path):
    with path.open(encoding="utf-8") as file:
        return json.load(file)


def read_jsonl(path):
    with path.open(encoding="utf-8") as file:
        for line in file:
            if line.strip():
                yield json.loads(line)


def show_table(rows):
    if not rows:
        print("No data available.")
        return
    columns = list(rows[0])
    lines = ["| " + " | ".join(columns) + " |",
             "| " + " | ".join("---" for _ in columns) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(row[column]) for column in columns) + " |")
    display(Markdown("\n".join(lines)))


def source_texts(paper):
    # Use the same source scope as prepare_qasper.py: abstract + body paragraphs.
    return [paper["abstract"]] + [
        text for section in paper["full_text"] for text in section["paragraphs"]
    ]


def describe(values):
    values = sorted(values)
    if not values:
        return {"count": 0, "min": None, "mean": None, "median": None, "p95": None, "max": None}
    return {
        "count": len(values), "min": values[0], "mean": round(mean(values), 2),
        "median": median(values),
        "p95": values[math.ceil(0.95 * len(values)) - 1],
        "max": values[-1],
    }


## Dataset sizes and blank sources

A paper contains multiple questions, and a question can have multiple human
answer annotations. Counts of annotations therefore differ from question counts.
Source entries below include abstracts and body paragraphs. Empty strings and
whitespace-only entries are counted separately; preprocessing skips both.


In [ ]:
raw = {split: load_json(RAW_DIR / f"qasper-{split}-v0.3.json") for split in SPLITS}
questions = {split: [q for paper in raw[split].values() for q in paper["qas"]]
             for split in SPLITS}
answers = {split: [a["answer"] for q in questions[split] for a in q["answers"]]
           for split in SPLITS}
texts = {split: [text for paper in raw[split].values() for text in source_texts(paper)]
         for split in SPLITS}

show_table([
    {"split": split, "papers": len(raw[split]), "questions": len(questions[split]),
     "answer annotations": len(answers[split]),
     "source entries": len(texts[split]),
     "empty strings": sum(text == "" for text in texts[split]),
     "whitespace-only": sum(text != "" and not text.strip() for text in texts[split])}
    for split in SPLITS
])


## Word lengths and questions per paper

Words are counted using whitespace splitting, consistent with the preprocessing
window boundaries. These are not embedding-model tokens. Blank source entries
are excluded from paragraph-length statistics. P95 uses the nearest-rank
percentile. The paragraph plot uses a logarithmic count axis to show the tail;
all source lengths are included.


In [ ]:
lengths = {
    split: {
        "source paragraph words": [len(text.split()) for text in texts[split] if text.strip()],
        "question words": [len(q["question"].split()) for q in questions[split]],
        "questions per paper": [len(paper["qas"]) for paper in raw[split].values()],
    }
    for split in SPLITS
}
show_table([{"split": split, "measurement": name, **describe(values)}
            for split in SPLITS for name, values in lengths[split].items()])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, measurement, title in zip(
    axes, ("source paragraph words", "question words"),
    ("Source paragraph lengths", "Question lengths"),
):
    # Both splits share bin boundaries for a meaningful comparison.
    maximum = max(max(lengths[split][measurement], default=0) for split in SPLITS)
    for split in SPLITS:
        axis.hist(lengths[split][measurement], bins=40, range=(0, max(1, maximum)),
                  histtype="step", linewidth=1.6, label=split)
    axis.set(title=title, xlabel="Words", ylabel="Count")
    axis.legend()
axes[0].set_yscale("log")
axes[0].set_ylabel("Count (log scale)")
fig.tight_layout()
plt.show()


## Answer types and answerability

Answer types are counted **per annotation**, not per question. For yes/no
answers, `false` means a valid **No** answer; only `null` means the field is unused.

Answerability is counted **per question**. A disagreement means some annotators
mark a question answerable and others mark it unanswerable. It does not measure
whether their answer wording differs.


In [ ]:
def answer_type(answer):
    if answer["unanswerable"]:
        return "unanswerable"
    if answer["yes_no"] is not None:
        return "yes/no"
    if answer["extractive_spans"]:
        return "extractive"
    if answer["free_form_answer"]:
        return "free-form"
    return "missing answer"


def question_answerability(question):
    flags = {a["answer"]["unanswerable"] for a in question["answers"]}
    if not flags:
        return "no annotations"
    if len(flags) > 1:
        return "disagreement"
    return "unanswerable" if True in flags else "answerable"


answer_counts = {split: Counter(answer_type(answer) for answer in answers[split])
                 for split in SPLITS}
answer_kinds = sorted({kind for counts in answer_counts.values() for kind in counts})
show_table([{"split": split, **{kind: answer_counts[split][kind] for kind in answer_kinds}}
            for split in SPLITS])

answerability = {split: Counter(question_answerability(q) for q in questions[split])
                for split in SPLITS}
show_table([{"split": split, **{kind: answerability[split][kind]
                               for kind in ("answerable", "unanswerable", "disagreement", "no annotations")}}
            for split in SPLITS])


In [ ]:
fig, axis = plt.subplots(figsize=(9, 4))
width = 0.36
for index, split in enumerate(SPLITS):
    percentages = [100 * answer_counts[split][kind] / len(answers[split]) for kind in answer_kinds]
    positions = [position + (index - 0.5) * width for position in range(len(answer_kinds))]
    axis.bar(positions, percentages, width=width, label=split)
axis.set_xticks(range(len(answer_kinds)), answer_kinds)
axis.set(title="Answer-type distribution", ylabel="Share of answer annotations (%)")
axis.legend()
fig.tight_layout()
plt.show()


## Evidence supplied by annotators

Evidence is a list of text strings attached to each answer annotation. Entries
can repeat across annotators. Figure/table selections begin with
`FLOAT SELECTED`; other entries can include paragraphs or section headings.
An empty evidence list is often associated with an unanswerable annotation.
Highlighted evidence is retained in the raw data but is not the paragraph-level
mapping target.


In [ ]:
evidence_counts = []
for split in SPLITS:
    entries = [entry for answer in answers[split] for entry in answer["evidence"]]
    evidence_counts.append({
        "split": split, "evidence entries": len(entries),
        "figure/table entries": sum(entry.strip().startswith("FLOAT SELECTED") for entry in entries),
        "blank entries": sum(not entry.strip() for entry in entries),
        "annotations with no evidence": sum(not answer["evidence"] for answer in answers[split]),
    })
show_table(evidence_counts)
show_table([{"split": split, "measurement": "evidence entries per annotation",
             **describe([len(answer["evidence"]) for answer in answers[split]])}
            for split in SPLITS])


## Inspect one original annotation

This example uses the first train paper and its first question/answer annotation.
It shows the original answer fields, evidence text, and highlighted evidence.
Change the selected paper or question here to explore another example.


In [ ]:
paper_id, paper = next(iter(raw["train"].items()))
question = paper["qas"][0]
example = {"paper_id": paper_id, "title": paper["title"],
           "question_id": question["question_id"], "question": question["question"],
           "annotation": question["answers"][0]}
print(json.dumps(example, ensure_ascii=False, indent=2))


## Processed chunk statistics (optional)

This section reads the generated corpus and preprocessing manifest. Missing
processed splits are skipped. Word counts come from actual chunk text; sizes
and overlap settings come from the matching preprocessing manifest. Regenerate
these files together when changing preprocessing settings.

The final histogram bin can have a spike because long paragraphs produce full-size
windows. With the default settings, many chunks are exactly 180 words, and
the last bin spans 174-180 words (including 180). Its left edge makes the
increase appear near 175; there is no separate 175-word cutoff.


In [ ]:
chunk_lengths = {}
processed_summary = []
for split in SPLITS:
    directory = PROCESSED_DIR / split
    if not (directory / "manifest.json").is_file() or not (directory / "corpus.jsonl").is_file():
        print(f"Skipping {split}: prepare the corpus to see chunk statistics.")
        continue
    manifest = load_json(directory / "manifest.json")
    chunk_lengths[split] = [len(chunk["text"].split()) for chunk in read_jsonl(directory / "corpus.jsonl")]
    processed_summary.append({
        "split": split, "retained sources": manifest["counts"]["paragraphs"],
        "chunks": len(chunk_lengths[split]),
        "configured words": manifest["chunking"]["size_words"],
        "overlap words": manifest["chunking"]["overlap_words"],
        "median words": median(chunk_lengths[split]) if chunk_lengths[split] else None,
        "max words": max(chunk_lengths[split], default=0),
    })
show_table(processed_summary)

if any(chunk_lengths.values()):
    fig, axis = plt.subplots(figsize=(9, 4))
    maximum = max(max(values, default=0) for values in chunk_lengths.values())
    for split, values in chunk_lengths.items():
        axis.hist(values, bins=30, range=(0, max(1, maximum)),
                  histtype="step", linewidth=1.6, label=split)
    axis.set(title="Processed chunk lengths", xlabel="Words", ylabel="Chunks")
    axis.legend()
    fig.tight_layout()
    plt.show()


## Gold-evidence mapping coverage (optional)

Counts below are calculated from `gold.jsonl`, not retrieval results. Every
question is retained. "Fully mapped" uses the current conservative
`all_evidence_mapped` flag: all annotations are answerable, each supplies
evidence, and every entry maps unambiguously to paragraph text.

Ambiguous matches retain candidate IDs but do not contribute automatic gold
positives. Figure/table evidence and section headings are audited because they
are not indexed as paragraphs. A question without mapped text evidence is not
necessarily an annotation error. Audit counts count entries across annotations,
not unique evidence paragraphs.


In [ ]:
mapping_summary = []
mapping_statuses = {}
for split in SPLITS:
    path = PROCESSED_DIR / split / "gold.jsonl"
    if not path.is_file():
        print(f"Skipping {split}: run evidence mapping to see coverage.")
        continue
    counts = Counter()
    statuses = Counter()
    for gold in read_jsonl(path):
        counts["questions"] += 1
        counts["with mapped text"] += bool(gold["paragraph_ids"])
        counts["fully mapped"] += gold["all_evidence_mapped"]
        for annotation in gold["annotations"]:
            statuses.update(entry["status"] for entry in annotation["evidence_mapping"])
    mapping_statuses[split] = statuses
    mapping_summary.append({"split": split, "questions": counts["questions"],
                            "with mapped text": counts["with mapped text"],
                            "fully mapped": counts["fully mapped"]})
show_table(mapping_summary)
status_names = sorted({status for counts in mapping_statuses.values() for status in counts})
show_table([{"split": split, **{status: counts[status] for status in status_names}}
            for split, counts in mapping_statuses.items()])


## Inspect mapping issues (optional)

The first three train audit entries are shown below. Evidence text is shortened
to 300 characters for display only. The original complete text remains in the
raw annotations and generated audit file.


In [ ]:
audit_path = PROCESSED_DIR / "train/evidence_audit.jsonl"
if audit_path.is_file():
    audit_examples = [{**entry, "text": entry["text"][:300]}
                      for entry in islice(read_jsonl(audit_path), 3)]
    print(json.dumps(audit_examples, ensure_ascii=False, indent=2))
else:
    print("Run evidence mapping on train to inspect audit examples.")


## Interpretation

- Keep references and evidence annotations outside retrieval indexes.
- Count gold paragraph evidence separately from the overlapping chunks derived
  from it; these plots and coverage counts do not define evaluation metrics.
- Define annotation/evidence eligibility rules before comparing retrieval systems
  and report excluded questions explicitly.
- Use train for retrieval development; keep held-out dev retrieval results out
  of parameter tuning. Aggregate data checks here do not evaluate retrieval.
- See [the preprocessing reference](../docs/preprocessing.md) for mapping rules.

Clear cell outputs before committing this notebook to avoid embedding dataset
examples and generated figures in Git.
